# Electoral Data Engineering


## Initialization

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

CSV_PATH    = "PRESIDENCIA_2024/CSV/2024_SEE_PRE_NAL_CAS.csv"
ELECTION_ID = "PRE_2024"

## Read Data

In [14]:
df_raw = pd.read_csv(CSV_PATH, low_memory=False, encoding="latin-1")
df_raw.columns = df_raw.columns.str.strip()

INT_COLS = [
    "ID_ESTADO", "ID_DISTRITO_FEDERAL", "ID_MUNICIPIO",
    "SECCION", "ID_CASILLA", "EXT_CONTIGUA",
    "LISTA_NOMINAL", "TOTAL_VOTOS", "NUM_VOTOS_VALIDOS",
    "NUM_VOTOS_NULOS", "NUM_VOTOS_CAN_NREG", "URNA_ELECTRONICA",
]
for col in INT_COLS:
    if col in df_raw.columns:
        df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")

print(f"Rows: {len(df_raw):,}  |  Columns: {df_raw.shape[1]}")
df_raw.head(3)

Rows: 170,766  |  Columns: 37


,ï»¿CIRCUNSCRIPCION,ID_ESTADO,NOMBRE_ESTADO,ID_DISTRITO_FEDERAL,CABECERA_DISTRITAL_FEDERAL,ID_MUNICIPIO,MUNICIPIO,SECCION,TIPO_CASILLA,ID_CASILLA,...,PT_MORENA,NUM_VOTOS_VALIDOS,NUM_VOTOS_CAN_NREG,NUM_VOTOS_NULOS,TOTAL_VOTOS,LISTA_NOMINAL,ESTATUS_ACTA,TRIBUNAL,RUTA_ACTA,OBSERVACIONES
0,2,1,AGUASCALIENTES,0,VOTO EN EL EXTRANJERO,0,VOTO EN EL EXTRANJERO,0,MEC,1,...,6,352,2,1,355,549,Cotejo (Levantada en Casilla),NaN,JE2024_01_PRE_M01_JCOTVMRE_248166.pdf,JE2024_01_PRE_M01_JCOTVMRE_248166.pdf
1,2,1,AGUASCALIENTES,0,VOTO EN EL EXTRANJERO,0,VOTO EN EL EXTRANJERO,0,MEC,1,...,9,1484,3,16,1503,1841,Cotejo (Levantada en Casilla),NaN,JE2024_01_PRE_V01_JCOTVMRE_E_248167.pdf,JE2024_01_PRE_V01_JCOTVMRE_E_248167.pdf
2,2,1,AGUASCALIENTES,0,VOTO EN EL EXTRANJERO,0,VOTO EN EL EXTRANJERO,0,MEC,2,...,1,237,1,0,238,138,Cotejo (Levantada en Casilla),NaN,JE2024_01_PRE_V02_JCOTVMRE_248168.pdf,JE2024_01_PRE_V02_JCOTVMRE_248168.pdf


## Table creation

### dim_election

In [15]:
dim_election = pd.DataFrame([{
    "election_id":   ELECTION_ID,
    "year":          2024,
    "election_type": "PRE",       # PRE / DIP_MR / DIP_RP / SEN_MR / SEN_RP / GOB
    "level":         "federal",
    "description":   "Elección Presidencial 2024",
}])

dim_election

,election_id,year,election_type,level,description
0,PRE_2024,2024,PRE,federal,Elección Presidencial 2024


### dim_geography

In [16]:
GEO_COLS = [
    "ID_ESTADO", "NOMBRE_ESTADO",
    "SECCION",
    "ID_MUNICIPIO", "MUNICIPIO",
    "ID_DISTRITO_FEDERAL", "CABECERA_DISTRITAL_FEDERAL",
]
# CIRCUNSCRIPCION is optional — include if present
if "CIRCUNSCRIPCION" in df_raw.columns:
    GEO_COLS.append("CIRCUNSCRIPCION")

dim_geography = (
    df_raw[GEO_COLS]
    .drop_duplicates()
    .dropna(subset=["ID_ESTADO", "SECCION"])
    .sort_values(["ID_ESTADO", "SECCION"])
    .reset_index(drop=True)
)

dim_geography["geo_id"] = (
    dim_geography["ID_ESTADO"].astype(int).astype(str).str.zfill(2)
    + "_"
    + dim_geography["SECCION"].astype(int).astype(str).str.zfill(4)
)

# Reorder so key is first
cols = ["geo_id"] + [c for c in dim_geography.columns if c != "geo_id"]
dim_geography = dim_geography[cols]

print(f"Unique sections: {len(dim_geography):,}")
dim_geography.head(3)

Unique sections: 70,536


,geo_id,ID_ESTADO,NOMBRE_ESTADO,SECCION,ID_MUNICIPIO,MUNICIPIO,ID_DISTRITO_FEDERAL,CABECERA_DISTRITAL_FEDERAL
0,01_0000,1,AGUASCALIENTES,0,0,VOTO EN EL EXTRANJERO,0,VOTO EN EL EXTRANJERO
1,01_0001,1,AGUASCALIENTES,1,1,AGUASCALIENTES,3,AGUASCALIENTES
2,01_0002,1,AGUASCALIENTES,2,1,AGUASCALIENTES,3,AGUASCALIENTES


### dim_casilla

In [17]:
CASILLA_COLS = [
    "ID_ESTADO", "SECCION", "ACTA_CASILLA-MEC",
    "TIPO_CASILLA", "ID_CASILLA", "EXT_CONTIGUA",   # kept as readable attributes
    "LISTA_NOMINAL", "URNA_ELECTRONICA",
    "ESTATUS_ACTA", "RUTA_ACTA",
]
available = [c for c in CASILLA_COLS if c in df_raw.columns]

dim_casilla = (
    df_raw[available]
    .drop_duplicates(subset=["ID_ESTADO", "SECCION", "ACTA_CASILLA-MEC"])
    .copy()
    .reset_index(drop=True)
)

dim_casilla["election_id"] = ELECTION_ID

# Primary key: nationally unique casilla identifier
dim_casilla["casilla_id"] = (
    dim_casilla["ID_ESTADO"].astype(int).astype(str).str.zfill(2)
    + "_"
    + dim_casilla["SECCION"].astype(int).astype(str).str.zfill(4)
    + "_"
    + dim_casilla["ACTA_CASILLA-MEC"].astype(str).str.strip()
)

# FK to dim_geography
dim_casilla["geo_id"] = (
    dim_casilla["ID_ESTADO"].astype(int).astype(str).str.zfill(2)
    + "_"
    + dim_casilla["SECCION"].astype(int).astype(str).str.zfill(4)
)

# Reorder so keys are first
cols = ["casilla_id", "election_id", "geo_id"] + [
    c for c in dim_casilla.columns
    if c not in ("casilla_id", "election_id", "geo_id")
]
dim_casilla = dim_casilla[cols]

print(f"Unique casillas: {len(dim_casilla):,}")
dim_casilla.head(3)

Unique casillas: 170,766


,casilla_id,election_id,geo_id,ID_ESTADO,SECCION,ACTA_CASILLA-MEC,TIPO_CASILLA,ID_CASILLA,EXT_CONTIGUA,LISTA_NOMINAL,URNA_ELECTRONICA,ESTATUS_ACTA,RUTA_ACTA
0,01_0000_VMRE01,PRE_2024,01_0000,1,0,VMRE01,MEC,1,0,549,1,Cotejo (Levantada en Casilla),JE2024_01_PRE_M01_JCOTVMRE_248166.pdf
1,01_0000_VeMRE01,PRE_2024,01_0000,1,0,VeMRE01,MEC,1,0,1841,1,Cotejo (Levantada en Casilla),JE2024_01_PRE_V01_JCOTVMRE_E_248167.pdf
2,01_0000_VeMRE02,PRE_2024,01_0000,1,0,VeMRE02,MEC,2,0,138,1,Cotejo (Levantada en Casilla),JE2024_01_PRE_V02_JCOTVMRE_248168.pdf


### dim_party

In [18]:
NON_PARTY_COLS = {
    "ï»¿CIRCUNSCRIPCION", "ID_ESTADO", "NOMBRE_ESTADO", "ID_DISTRITO_FEDERAL",
    "CABECERA_DISTRITAL_FEDERAL", "ID_MUNICIPIO", "MUNICIPIO", "SECCION",
    "TIPO_CASILLA", "ID_CASILLA", "EXT_CONTIGUA", "ACTA_CASILLA-MEC",
    "URNA_ELECTRONICA", "NUM_VOTOS_VALIDOS", "NUM_VOTOS_CAN_NREG",
    "NUM_VOTOS_NULOS", "TOTAL_VOTOS", "LISTA_NOMINAL", "ESTATUS_ACTA",
    "TRIBUNAL", "RUTA_ACTA", "OBSERVACIONES",
}

party_keys = [
    c for c in df_raw.columns
    if c not in NON_PARTY_COLS
    and pd.api.types.is_numeric_dtype(df_raw[c])
]

PARTY_META = {
    "MORENA":         {"short_name": "Morena",        "is_coalition": False, "members": []},
    "PT":             {"short_name": "PT",             "is_coalition": False, "members": []},
    "PVEM":           {"short_name": "PVEM",           "is_coalition": False, "members": []},
    "PAN":            {"short_name": "PAN",            "is_coalition": False, "members": []},
    "PRI":            {"short_name": "PRI",            "is_coalition": False, "members": []},
    "PRD":            {"short_name": "PRD",            "is_coalition": False, "members": []},
    "MC":             {"short_name": "MC",             "is_coalition": False, "members": []},
    "PAN_PRI_PRD":    {"short_name": "PAN+PRI+PRD",    "is_coalition": True,  "members": ["PAN","PRI","PRD"]},
    "PAN_PRI":        {"short_name": "PAN+PRI",        "is_coalition": True,  "members": ["PAN","PRI"]},
    "PAN_PRD":        {"short_name": "PAN+PRD",        "is_coalition": True,  "members": ["PAN","PRD"]},
    "PRI_PRD":        {"short_name": "PRI+PRD",        "is_coalition": True,  "members": ["PRI","PRD"]},
    "PVEM_PT_MORENA": {"short_name": "PVEM+PT+Morena", "is_coalition": True,  "members": ["PVEM","PT","MORENA"]},
    "PVEM_PT":        {"short_name": "PVEM+PT",        "is_coalition": True,  "members": ["PVEM","PT"]},
    "PVEM_MORENA":    {"short_name": "PVEM+Morena",    "is_coalition": True,  "members": ["PVEM","MORENA"]},
    "PT_MORENA":      {"short_name": "PT+Morena",      "is_coalition": True,  "members": ["PT","MORENA"]},
}

dim_party = pd.DataFrame([
    {
        "party_key":    key,
        "is_coalition": PARTY_META.get(key, {"is_coalition": False})["is_coalition"],
        "members":      ",".join(PARTY_META.get(key, {"members": []})["members"]),
    }
    for key in party_keys
])

print(f"Parties / coalitions found: {len(dim_party)}")
dim_party

Parties / coalitions found: 15


,party_key,is_coalition,members
0,PAN,False,
1,PRI,False,
2,PRD,False,
3,PVEM,False,
4,PT,False,
5,MC,False,
6,MORENA,False,
7,PAN_PRI_PRD,True,"PAN,PRI,PRD"
8,PAN_PRI,True,"PAN,PRI"
9,PAN_PRD,True,"PAN,PRD"


### fact_casilla_vote

In [19]:
# Build casilla_id on df_raw using the same logic as dim_casilla
df_raw["casilla_id"] = (
    df_raw["ID_ESTADO"].astype(int).astype(str).str.zfill(2)
    + "_"
    + df_raw["SECCION"].astype(int).astype(str).str.zfill(4)
    + "_"
    + df_raw["ACTA_CASILLA-MEC"].astype(str).str.strip()
)

VOTE_META_COLS = [
    "NUM_VOTOS_VALIDOS", "NUM_VOTOS_NULOS",
    "NUM_VOTOS_CAN_NREG", "TOTAL_VOTOS",
]

fact_casilla_vote = (
    df_raw[["casilla_id"] + party_keys + VOTE_META_COLS]
    .melt(
        id_vars=["casilla_id"] + VOTE_META_COLS,
        value_vars=party_keys,
        var_name="party_key",
        value_name="votes",
    )
)

fact_casilla_vote["election_id"] = ELECTION_ID
fact_casilla_vote["votes"]       = fact_casilla_vote["votes"].fillna(0).astype(int)

fact_casilla_vote = fact_casilla_vote[[
    "election_id", "casilla_id", "party_key", "votes",
    "NUM_VOTOS_VALIDOS", "NUM_VOTOS_NULOS", "NUM_VOTOS_CAN_NREG", "TOTAL_VOTOS",
]]

print(f"Fact rows: {len(fact_casilla_vote):,}")
fact_casilla_vote.head(6)

Fact rows: 2,561,490


,election_id,casilla_id,party_key,votes,NUM_VOTOS_VALIDOS,NUM_VOTOS_NULOS,NUM_VOTOS_CAN_NREG,TOTAL_VOTOS
0,PRE_2024,01_0000_VMRE01,PAN,88,352,1,2,355
1,PRE_2024,01_0000_VeMRE01,PAN,684,1484,16,3,1503
2,PRE_2024,01_0000_VeMRE02,PAN,102,237,0,1,238
3,PRE_2024,01_0338_B,PAN,69,413,12,0,425
4,PRE_2024,01_0338_C01,PAN,74,391,21,0,412
5,PRE_2024,01_0338_C02,PAN,68,384,20,0,404


## Sanity checks

In [20]:
# 1. Vote totals by party
print("── Votes by party ──────────────────────────────")
totals = (
    fact_casilla_vote
    .groupby("party_key")["votes"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
print(totals.to_string(index=False))

# 2. FK: every casilla in fact exists in dim_casilla
orphan_casillas = set(fact_casilla_vote["casilla_id"]) - set(dim_casilla["casilla_id"])
print(f"\nOrphan casilla_ids : {len(orphan_casillas)}")

# 3. FK: every party in fact exists in dim_party
orphan_parties = set(fact_casilla_vote["party_key"]) - set(dim_party["party_key"])
print(f"Orphan party_keys  : {len(orphan_parties)}")

# 4. FK: every geo_id in dim_casilla exists in dim_geography
orphan_geos = set(dim_casilla["geo_id"]) - set(dim_geography["geo_id"])
print(f"Orphan geo_ids     : {len(orphan_geos)}")

# 5. TOTAL_VOTOS integrity — should equal validos + nulos + can_nreg
sample = df_raw.copy()
sample["check"] = (
    sample["NUM_VOTOS_VALIDOS"] +
    sample["NUM_VOTOS_NULOS"] +
    sample["NUM_VOTOS_CAN_NREG"]
)
mismatches = (sample["check"] != sample["TOTAL_VOTOS"]).sum()
print(f"TOTAL_VOTOS mismatches: {mismatches:,}")

── Votes by party ──────────────────────────────
     party_key    votes
        MORENA 26253825
           PAN  9224341
            MC  6204710
           PRI  5320727
          PVEM  3687773
            PT  2878024
PVEM_PT_MORENA  2041403
   PAN_PRI_PRD   884579
           PRD   793603
     PT_MORENA   445664
   PVEM_MORENA   414515
       PAN_PRI   213807
       PVEM_PT   203315
       PAN_PRD    37277
       PRI_PRD    28363

Orphan casilla_ids : 0
Orphan party_keys  : 0
Orphan geo_ids     : 0
TOTAL_VOTOS mismatches: 0


## Write data

In [21]:
out = Path("data/clean")
out.mkdir(parents=True, exist_ok=True)

dim_election.to_parquet(out / "dim_election.parquet",   index=False)
dim_geography.to_parquet(out / "dim_geography.parquet", index=False)
dim_casilla.to_parquet(out / "dim_casilla.parquet",     index=False)
dim_party.to_parquet(out / "dim_party.parquet",         index=False)

fact_casilla_vote.to_parquet(
    out / "fact_casilla_vote.parquet",
    index=False,
    partition_cols=["election_id"],
)

print("Done →", out.resolve())

Done → /Users/efrenzagal/Documents/GitHub/current-affairs-mx-elections-pres-2024/data/clean
